In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
   
)
roc_curve,
roc_auc_score

os.makedirs("charts", exist_ok=True)
def load_data():
    """
    Load cleaned Titanic dataset.
    """

    print("=" * 60)
    print("LOADING CLEANED DATASET")
    print("=" * 60)

    df = pd.read_csv("titanic.csv")

    print("\nDataset Loaded Successfully.")
    print(f"Shape : {df.shape}")

    return df


def prepare_data(df):
    """
    Prepare features and target.
    Remove columns that cause target leakage.
    """

    print("\n" + "=" * 60)
    print("FEATURE SELECTION")
    print("=" * 60)

    target = "survived"

    # Remove target and leakage column
    X = df.drop(columns=["survived", "alive"])

    y = df[target]

    print(f"\nTarget Column : {target}")
    print(f"Features      : {X.shape[1]}")

    print("\nRemoved Columns")
    print("- survived (Target)")
    print("- alive (Target Leakage)")

    return X, y

def split_data(X, y):
    """
    Stratified Train-Test Split.
    """

    print("\n" + "=" * 60)
    print("TRAIN TEST SPLIT")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print(f"\nTraining Samples : {len(X_train)}")
    print(f"Testing Samples  : {len(X_test)}")

    return X_train, X_test, y_train, y_test


def create_preprocessor(X_train):
    """
    Create preprocessing pipeline.
    """

    print("\n" + "=" * 60)
    print("PREPROCESSING")
    print("=" * 60)

    numeric_features = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    print("\nNumeric Features")
    print(numeric_features)

    print("\nCategorical Features")
    print(categorical_features)

    numeric_transformer = Pipeline(
        steps=[
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore")
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_transformer,
                numeric_features
            ),
            (
                "cat",
                categorical_transformer,
                categorical_features
            )
        ]
    )

    print("\nPreprocessing Pipeline Created Successfully.")

    return preprocessor
def evaluate_model(model_name, model, X_test, y_test):
    """
    Evaluate Classification Model
    """

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n" + "=" * 60)
    print(model_name.upper())
    print("=" * 60)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")

    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()

    plt.savefig(
        f"charts/{model_name.lower().replace(' ', '_')}_cm.png"
    )

    plt.close()

    print("Confusion Matrix Saved.")
    fpr, tpr, auc = get_roc_data(
    model,
    X_test,
    y_test
    )
    return {
    "Model": model_name,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "AUC": auc,
    "FPR": fpr,
    "TPR": tpr
}
def get_roc_data(model, X_test, y_test):
    """
    Return ROC data and AUC for a trained model.
    """

    y_prob = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    auc = roc_auc_score(y_test, y_prob)

    return fpr, tpr, auc

def train_logistic_regression(
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test
):
    """
    Train Logistic Regression Model
    """

    print("\n" + "=" * 60)
    print("TRAINING LOGISTIC REGRESSION")
    print("=" * 60)

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    print("Training Completed Successfully.")

    results = evaluate_model(
        "Logistic Regression",
        model,
        X_test,
        y_test
    )

    return model, results   

def train_decision_tree(
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test
):
    """
    Train Decision Tree Model
    """

    print("\n" + "=" * 60)
    print("TRAINING DECISION TREE")
    print("=" * 60)

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                DecisionTreeClassifier(
                    random_state=42,
                    max_depth=5
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    print("Training Completed Successfully.")

    results = evaluate_model(
        "Decision Tree",
        model,
        X_test,
        y_test
    )

    return model, results

def visualize_decision_tree(model):
    """
    Visualize Decision Tree using plot_tree()
    """

    print("\n" + "=" * 60)
    print("DECISION TREE VISUALIZATION")
    print("=" * 60)

    # Get trained tree
    tree_model = model.named_steps["classifier"]

    # Get feature names after preprocessing
    feature_names = model.named_steps[
        "preprocessor"
    ].get_feature_names_out()

    plt.figure(figsize=(28, 14))

    plot_tree(
        decision_tree=tree_model,
        feature_names=feature_names,
        class_names=["Not Survived", "Survived"],
        filled=True,
        rounded=True,
        fontsize=8
    )

    plt.title("Decision Tree Classifier")

    plt.tight_layout()

    plt.savefig(
        "charts/decision_tree.png",
        dpi=300
    )

    plt.close()

    print("✓ Decision Tree saved inside charts/")

def plot_roc_comparison(results):
    """
    Plot ROC curves for all models in one figure.
    """

    plt.figure(figsize=(8, 6))

    for result in results:
        plt.plot(
            result["FPR"],
            result["TPR"],
            linewidth=2,
            label=f'{result["Model"]} (AUC={result["AUC"]:.3f})'
        )

    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Comparison")

    plt.legend()

    plt.tight_layout()

    plt.savefig("charts/roc_comparison.png", dpi=300)

    plt.close()

    print("✓ ROC Comparison saved inside charts/")

def train_random_forest(
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test
):
    """
    Train Random Forest Model
    """

    print("\n" + "=" * 60)
    print("TRAINING RANDOM FOREST")
    print("=" * 60)

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=100,
                    random_state=42
                )
            )
        ]
    )

    model.fit(X_train, y_train)

    print("Training Completed Successfully.")

    results = evaluate_model(
        "Random Forest",
        model,
        X_test,
        y_test
    )

    return model, results
def compare_imbalance_methods(
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test
):
    """
    Compare Baseline, Balanced and SMOTE models.
    """

    print("\n" + "=" * 60)
    print("TASK 11 - IMBALANCE HANDLING")
    print("=" * 60)

    results = []

    # Baseline Model
    baseline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier",
         RandomForestClassifier(
             random_state=42
         ))
    ])

    baseline.fit(X_train, y_train)

    pred = baseline.predict(X_test)

    results.append({
        "Method": "Baseline",
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

    # Balanced Model
    balanced = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier",
         RandomForestClassifier(
             class_weight="balanced",
             random_state=42
         ))
    ])

    balanced.fit(X_train, y_train)

    pred = balanced.predict(X_test)

    results.append({
        "Method": "Balanced",
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

    # SMOTE Model
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    smote = SMOTE(random_state=42)

    X_smote, y_smote = smote.fit_resample(
        X_train_processed,
        y_train
    )

    rf_smote = RandomForestClassifier(
        random_state=42
    )

    rf_smote.fit(
        X_smote,
        y_smote
    )

    pred = rf_smote.predict(
        X_test_processed
    )

    results.append({
        "Method": "SMOTE",
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

    # Create comparison DataFrame
    comparison = pd.DataFrame(results)

    print()
    print(comparison.to_string(index=False))

    # Find best method
    best = comparison.loc[
        comparison["F1"].idxmax(),
        "Method"
    ]

    print("\nConclusion")
    print(f"Best Method based on F1 Score : {best}")

    # IMPORTANT: Return the DataFrame
    return comparison
def tune_random_forest(
    preprocessor,
    X_train,
    y_train
):
    """
    Task 12
    Hyperparameter Tuning using GridSearchCV
    """

    print("\n" + "=" * 60)
    print("TASK 12 - GRIDSEARCHCV")
    print("=" * 60)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42
        ))
    ])

    param_grid = {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [5, 10, None],
        "classifier__min_samples_split": [2, 5]
    }

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("\nBest Parameters")
    print(grid.best_params_)

    print("\nBest Cross Validation F1 Score")
    print(f"{grid.best_score_:.4f}")

    return grid.best_estimator_

def calculate_oob_score(
    preprocessor,
    X_train,
    y_train
):
    """
    Calculate Out-of-Bag Score
    """

    print("\n" + "=" * 60)
    print("OUT OF BAG SCORE")
    print("=" * 60)

    X_train_processed = preprocessor.fit_transform(X_train)

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        oob_score=True
    )

    rf.fit(
        X_train_processed,
        y_train
    )

    print(f"OOB Score : {rf.oob_score_:.4f}")

def regression_analysis(df):
    """
    Task 13
    Linear Regression Analysis
    """

    print("\n" + "=" * 60)
    print("TASK 13 - LINEAR REGRESSION")
    print("=" * 60)

    # --------------------------
    # Predict Fare
    # --------------------------

    X = df.drop(columns=["fare"])

    y = df["fare"]

    categorical = X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns

    X = pd.get_dummies(
        X,
        columns=categorical,
        drop_first=True
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    model = LinearRegression()

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    n = len(y_test)

    p = X_test.shape[1]

    adjusted_r2 = 1 - (
        (1-r2)*(n-1)/(n-p-1)
    )

    print(f"\nMAE         : {mae:.4f}")
    print(f"RMSE        : {rmse:.4f}")
    print(f"R² Score    : {r2:.4f}")
    print(f"Adjusted R² : {adjusted_r2:.4f}")

    # --------------------------
    # Residual Plot
    # --------------------------

    residuals = y_test - prediction

    plt.figure(figsize=(7,5))

    plt.scatter(
        prediction,
        residuals
    )

    plt.axhline(
        y=0,
        color="red",
        linestyle="--"
    )

    plt.xlabel("Predicted Fare")

    plt.ylabel("Residual")

    plt.title("Residual Plot")

    plt.tight_layout()

    plt.savefig(
        "charts/residual_plot.png"
    )

    plt.close()

    print("✓ Residual Plot Saved.")
    # --------------------------
    # Heteroscedasticity Interpretation
    # --------------------------
    print("\nHeteroscedasticity Interpretation:")
    print(
        "The residual plot shows an increasing spread of residuals as "
        "predicted fare increases. This indicates the presence of "
        "heteroscedasticity, meaning the error variance is not constant."
    )


    return model  

def final_summary(
    comparison_df,
    imbalance_df
):
    """
    Task 14
    Final Model Comparison
    """

    print("\n" + "=" * 60)
    print("TASK 14 - FINAL SUMMARY")
    print("=" * 60)

    print("\nClassification Models")
    print(comparison_df.to_string(index=False))

    best_classifier = comparison_df.loc[
        comparison_df["F1 Score"].idxmax()
    ]

    print("\nBest Classification Model")
    print(best_classifier["Model"])

    print("\nBest Imbalance Handling Method")

    best_method = imbalance_df.loc[
        imbalance_df["F1"].idxmax()
    ]

    print(best_method["Method"])

    print("\nOverall Recommendation")

    print(
        f"""
1. Logistic Regression achieved the best overall classification performance.

2. Balanced Class Weight produced the best F1 Score among imbalance handling techniques.

3. Random Forest performed well after hyperparameter tuning.

4. Linear Regression can be used for Fare prediction, but the R² score indicates only moderate predictive performance.
"""
    )  
import joblib

def save_and_load_model(model):

    print("\n" + "=" * 60)
    print("TASK 15 - MODEL PERSISTENCE")
    print("=" * 60)

    joblib.dump(model, "best_model.pkl")

    print("Model Saved Successfully.")

    loaded_model = joblib.load("best_model.pkl")

    print("Model Loaded Successfully.")

    return loaded_model

def main():

    # Load Dataset
    df = load_data()

    # Prepare Features and Target
    X, y = prepare_data(df)

    # Train-Test Split
    X_train, X_test, y_train, y_test = split_data(
        X,
        y
    )

    # Create Preprocessing Pipeline
    preprocessor = create_preprocessor(X_train)

    print("\n" + "=" * 60)
    print("TRAINING MACHINE LEARNING MODELS")
    print("=" * 60)

    # Logistic Regression
    logistic_model, logistic_results = train_logistic_regression(
        preprocessor,
        X_train,
        X_test,
        y_train,
        y_test
    )

    # Decision Tree
    decision_tree_model, tree_results = train_decision_tree(
        preprocessor,
        X_train,
        X_test,
        y_train,
        y_test
    )

    visualize_decision_tree(decision_tree_model)

    # Random Forest
    random_forest_model, forest_results = train_random_forest(
        preprocessor,
        X_train,
        X_test,
        y_train,
        y_test
    )

    results = [
        logistic_results,
        tree_results,
        forest_results
    ]

    plot_roc_comparison(results)

    comparison_df = pd.DataFrame([
        {
            "Model": r["Model"],
            "Accuracy": r["Accuracy"],
            "Precision": r["Precision"],
            "Recall": r["Recall"],
            "F1 Score": r["F1 Score"],
            "AUC": r["AUC"]
        }
        for r in results
    ])

    print("\n" + "=" * 60)
    print("MODEL COMPARISON")
    print("=" * 60)

    print(comparison_df.to_string(index=False))

    # Task 11
    imbalance_df = compare_imbalance_methods(
        preprocessor,
        X_train,
        X_test,
        y_train,
        y_test
    )

    # Task 12
    best_model = tune_random_forest(
        preprocessor,
        X_train,
        y_train
    )

    calculate_oob_score(
        preprocessor,
        X_train,
        y_train
    )

    # Task 13
    regression_analysis(df)

    # Task 14
    final_summary(
        comparison_df,
        imbalance_df
    )

    # Task 15
    save_and_load_model(best_model)


if __name__ == "__main__":
    main()

LOADING CLEANED DATASET

Dataset Loaded Successfully.
Shape : (889, 14)

FEATURE SELECTION

Target Column : survived
Features      : 12

Removed Columns
- survived (Target)
- alive (Target Leakage)

TRAIN TEST SPLIT

Training Samples : 711
Testing Samples  : 178

PREPROCESSING

Numeric Features
['pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical Features
['sex', 'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alone']

Preprocessing Pipeline Created Successfully.

TRAINING MACHINE LEARNING MODELS

TRAINING LOGISTIC REGRESSION
Training Completed Successfully.

LOGISTIC REGRESSION
Accuracy  : 0.8146
Precision : 0.7966
Recall    : 0.6912
F1 Score  : 0.7402
Confusion Matrix Saved.

TRAINING DECISION TREE
Training Completed Successfully.

DECISION TREE
Accuracy  : 0.7921
Precision : 0.8163
Recall    : 0.5882
F1 Score  : 0.6838
Confusion Matrix Saved.

DECISION TREE VISUALIZATION
✓ Decision Tree saved inside charts/

TRAINING RANDOM FOREST
Training Completed Successfully.

RAND